
# FMCG End-to-End Data Engineering Project --- Databricks

## 📌 Project Overview

This project demonstrates an **end-to-end data engineering pipeline
using Databricks, PySpark, Delta Lake, Unity Catalog, and Amazon S3**
for an FMCG parent company and its newly acquired child company,
**Sportsbar**.

The parent FMCG company already has an established data platform and
production pipeline. When the company acquired Sportsbar, the acquired
company had source data in S3 but **did not have an existing data
pipeline** for data cleansing, dimensional processing, fact processing,
and integration with the parent company's analytical model.

The objective of this project is to build the missing data engineering
pipeline for Sportsbar and integrate its data into the existing
parent-company Gold layer.

------------------------------------------------------------------------

## 🎯 Business Requirement

The parent company already maintains:

-   Existing Bronze, Silver, and Gold layers
-   Existing dimension tables in the Gold layer
-   Existing pipeline/job orchestration
-   Existing analytical data model

After acquiring Sportsbar, the child company's data needs to be
onboarded into the existing platform.

The solution therefore needs to:

1.  Extract Sportsbar source data from Amazon S3.
2.  Register and process the data through Unity Catalog.
3.  Store raw data in the **Bronze** layer.
4.  Clean and standardize data in the **Silver** layer.
5.  Process child-company dimension data.
6.  Merge Sportsbar dimension records into the existing parent-company
    Gold dimension tables.
7.  Develop a **full-load fact pipeline** for Sportsbar.
8.  Develop an **incremental-load fact pipeline** for Sportsbar.
9.  Merge/update new and changed records appropriately.
10. Schedule the complete pipeline for automated execution.

------------------------------------------------------------------------

## 🏗️ Architecture

``` text
                         ┌───────────────────────┐
                         │      Amazon S3         │
                         │ Sportsbar Source CSVs │
                         └───────────┬───────────┘
                                     │
                                     ▼
                         ┌───────────────────────┐
                         │     Databricks        │
                         │  PySpark / Spark SQL  │
                         └───────────┬───────────┘
                                     │
                                     ▼
                    ┌────────────────────────────────┐
                    │ Unity Catalog                  │
                    │ Catalog: fmcg                   │
                    └────────────────┬───────────────┘
                                     │
                                     ▼
                         ┌───────────────────────┐
                         │      BRONZE           │
                         │ Raw source data       │
                         │ customers             │
                         │ products              │
                         │ orders                │
                         │ gross_price           │
                         └───────────┬───────────┘
                                     │
                              Data Cleansing
                              & Standardization
                                     │
                                     ▼
                         ┌───────────────────────┐
                         │       SILVER          │
                         │ Cleaned/standardized  │
                         │ Sportsbar data        │
                         └───────────┬───────────┘
                                     │
                         ┌───────────┴───────────┐
                         │                       │
                         ▼                       ▼
                ┌─────────────────┐     ┌─────────────────┐
                │ Dimension       │     │ Fact Processing │
                │ Processing      │     │ Full Load       │
                │ + MERGE         │     │ Incremental     │
                └────────┬────────┘     └────────┬────────┘
                         │                       │
                         └───────────┬───────────┘
                                     ▼
                         ┌───────────────────────┐
                         │        GOLD           │
                         │ Parent + Sportsbar    │
                         │ Integrated Data Model │
                         └───────────┬───────────┘
                                     │
                                     ▼
                         ┌───────────────────────┐
                         │ Jobs / Workflows      │
                         │ Scheduling & Automation│
                         └───────────────────────┘
```

------------------------------------------------------------------------

## 🧱 Medallion Architecture

### 🥉 Bronze Layer

The Bronze layer stores the source data extracted from the Sportsbar S3
location.

Example source datasets:

-   `customers`
-   `products`
-   `orders`
-   `gross_price`

The source files are read from an S3 path similar to:

``` text
s3://sportsbar-dp-sh/<data_source>/*.csv
```

The source location and data source are parameterized using Databricks
widgets.

Example:

``` python
dbutils.widgets.text("catalog", "fmcg")
dbutils.widgets.text("data_source", "customers")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f"s3://sportsbar-dp-sh/{data_source}/*.csv"
```

------------------------------------------------------------------------

### 🥈 Silver Layer

The Silver layer contains cleaned and standardized data.

Typical transformations include:

-   Removing duplicate records
-   Trimming unwanted spaces
-   Standardizing text
-   Standardizing city names
-   Handling invalid or inconsistent values
-   Applying business rules
-   Preparing data for dimension and fact processing

Example:

``` python
df_silver = df_silver.dropDuplicates(["customer_id"])

df_silver = df_silver.withColumn(
    "customer_name",
    F.trim(F.col("customer_name"))
)

df_silver = df_silver.withColumn(
    "customer_name",
    F.initcap(F.col("customer_name"))
)
```

For example, inconsistent city values such as:

``` text
Hyderbad
Hyderabadd
NewDelhee
Bengaluru
Bengalore
```

can be standardized to business-approved values.

------------------------------------------------------------------------

### 🥇 Gold Layer

The Gold layer contains business-ready dimensional and fact tables.

The parent company already has Gold dimension tables such as:

``` text
dim_customers
dim_date
dim_gross_price
dim_products
```

and fact data such as:

``` text
fact_orders
```

Sportsbar-specific processing is performed and then integrated with the
parent company's existing Gold model.

------------------------------------------------------------------------

# 🔄 Dimension Data Integration

One of the key requirements of this project is integrating Sportsbar's
dimension data into the existing parent-company dimension tables.

For example:

``` text
Sportsbar Silver
      │
      ▼
Sportsbar Dimension Processing
      │
      ▼
Data Validation / Transformation
      │
      ▼
MERGE
      │
      ▼
Parent Gold Dimension
```

The `MERGE` operation is used to implement **insert and update logic**.

Conceptually:

``` sql
MERGE INTO fmcg.gold.dim_customers AS target
USING sportsbar_dimension AS source
ON target.customer_id = source.customer_id

WHEN MATCHED THEN
  UPDATE SET *

WHEN NOT MATCHED THEN
  INSERT *
```

### MERGE Behavior

  Scenario                                     Action
  -------------------------------------------- ----------------------
  Existing business key + changed attributes   Update target
  New business key                             Insert new record
  Existing record with no change               Keep existing record

This allows Sportsbar data to become part of the parent company's
existing dimensional model without rebuilding the entire dimension table
every time.

------------------------------------------------------------------------

# 📦 Fact Data Processing

Sportsbar fact data requires two different processing approaches.

## 1. Full Load

The full-load pipeline is used for the initial ingestion of Sportsbar
fact data.

Flow:

``` text
S3
 ↓
Bronze
 ↓
Silver
 ↓
Transformations
 ↓
Gold Fact Table
```

The full-load process establishes the initial historical dataset in the
Gold fact table.

Example notebook:

``` text
1_full_load_fact
```

------------------------------------------------------------------------

## 2. Incremental Load

After the initial full load, processing the entire source dataset
repeatedly would be inefficient.

Therefore, an incremental pipeline is developed to process only new or
changed data.

Flow:

``` text
New/Changed S3 Data
        ↓
      Bronze
        ↓
      Silver
        ↓
Incremental Detection
        ↓
Business Transformations
        ↓
Gold Fact Table
```

Example notebook:

``` text
2_incremental_load_fact
```

The incremental process can use an appropriate business key, timestamp,
watermark, or source-system change indicator depending on the source
data design.

------------------------------------------------------------------------

# 📅 Job Scheduling, Orchestration and Notifications

After development and testing, the individual notebooks are organized into Databricks Workflows/Jobs.

For the Sportsbar incremental processing, a dedicated Databricks job named **`fmcg_incremental_job_processing`** has been created with the following task dependency flow:

```text
dim_processing_customers
        ↓
dim_processing_products
        ↓
dim_processing_pricing
        ↓
fact_processing_orders
```

All tasks are configured to run using **Databricks Serverless compute**.

### Incremental Job Schedule

The incremental job is scheduled to run **every day at 11:00 PM**.

```text
Frequency : Daily
Time      : 11:00 PM
Purpose   : Process new/changed Sportsbar data
Compute   : Serverless
```

### Email Notifications

Email notifications are configured for the relevant project/team members so they are informed about the job execution status. Notifications provide visibility for successful completion, failures, and execution issues requiring attention.

The project contains a consolidated pipeline structure:

```text
consolidated_pipeline/
│
├── 1_setup/
│   └── utilities
│
├── 2_dimension_data_processing/
│
└── 3_fact_data_processing/
```

The workflow is designed to execute the processing in the correct order.
------------------------------------------------------------------------

# 🗂️ Unity Catalog Structure

The project uses Unity Catalog to organize the data platform.

Example:

``` text
Catalog
└── fmcg
    │
    ├── bronze
    │   ├── customers
    │   ├── products
    │   ├── orders
    │   └── gross_price
    │
    ├── silver
    │   ├── customers
    │   ├── products
    │   ├── orders
    │   └── gross_price
    │
    └── gold
        ├── dim_customers
        ├── dim_date
        ├── dim_gross_price
        ├── dim_products
        ├── fact_orders
        └── Sportsbar integrated data
```

------------------------------------------------------------------------

# 🪣 External S3 Data Location

Sportsbar source data is maintained in an S3 external location.

Example:

``` text
s3://sportsbar-dp-sh/
```

The project uses a Databricks external location and credential
configuration to access the S3 data securely.

Example source structure:

``` text
sportsbar-dp-sh/
│
├── customers/
│   └── *.csv
│
├── products/
│   └── *.csv
│
├── orders/
│   └── *.csv
│
└── gross_price/
    └── *.csv
```

------------------------------------------------------------------------

# 🧰 Technologies Used

  Technology                 Purpose
  -------------------------- --------------------------------------------------
  **Databricks**             Data engineering and pipeline development
  **PySpark**                Data extraction and transformation
  **Spark SQL**              SQL-based data processing
  **Delta Lake**             Reliable analytical storage and MERGE operations
  **Unity Catalog**          Data governance and organization
  **Amazon S3**              Source data storage
  **Databricks Workflows**   Job orchestration and scheduling
  **Python**                 Pipeline development and utility logic
  **SQL**                    Data transformation and validation

------------------------------------------------------------------------

# 📁 Project Structure

``` text
consolidated_pipeline/
│
├── 1_setup/
│   └── utilities
│
├── 2_dimension_data_processing/
│   ├── dimension_customer_processing
│   ├── dimension_product_processing
│   └── other_dimension_processing
│
└── 3_fact_data_processing/
    ├── 1_full_load_fact
    └── 2_incremental_load_fact
```

Additional notebooks include:

``` text
employee_and_department
utilities
dimension_date_processing
1_customer_data
```

------------------------------------------------------------------------

# 🔧 Key Engineering Concepts Demonstrated

This project demonstrates practical data engineering concepts including:

-   End-to-end ETL/ELT pipeline development
-   Amazon S3 ingestion
-   Databricks
-   PySpark
-   Spark SQL
-   Unity Catalog
-   Medallion Architecture
-   Bronze/Silver/Gold layers
-   Delta Lake
-   Data cleansing
-   Data standardization
-   Duplicate removal
-   Parameterized notebooks
-   Databricks widgets
-   Dimension processing
-   Fact processing
-   Full load
-   Incremental load
-   Delta `MERGE`
-   Insert and update processing
-   Data validation
-   Job orchestration
-   Workflow scheduling
-   Parent-child company data integration

------------------------------------------------------------------------

# 🔍 Data Quality and Transformation

The pipeline applies data-quality rules before loading data into the
Gold layer.

Examples include:

``` text
Source Data
    ↓
Duplicate Check
    ↓
Null/Invalid Value Handling
    ↓
Trim & Standardize
    ↓
Business Rule Validation
    ↓
Dimension/Fact Transformation
    ↓
Gold
```

This ensures that the acquired company's data follows the parent
company's expected data standards.

------------------------------------------------------------------------

# 🚀 End-to-End Flow

### Daily Incremental Processing

```text
Daily 11:00 PM Schedule
        │
        ▼
dim_processing_customers
        │
        ▼
dim_processing_products
        │
        ▼
dim_processing_pricing
        │
        ▼
fact_processing_orders
        │
        ▼
Email Notification
        │
        ▼
Relevant Team Members
```

The complete project can be summarized as:

``` text
Sportsbar S3
     │
     ▼
External Location / Credential
     │
     ▼
Databricks
     │
     ▼
Bronze
Raw CSV Data
     │
     ▼
Silver
Clean + Standardize
     │
     ├──────────────────────┐
     ▼                      ▼
Dimension Processing     Fact Processing
     │                      │
     ▼                      ├── Full Load
MERGE                      │
     │                      └── Incremental Load
     ▼                             │
Parent Gold Dimensions            ▼
     │                       Parent Gold Facts
     └──────────────┬──────────────┘
                    ▼
             Integrated Gold
                    │
                    ▼
             Databricks Job
                    │
                    ▼
               Scheduling
```

------------------------------------------------------------------------

# 💡 Real-World Scenario

This project represents a common enterprise data engineering
requirement:

> A large parent organization acquires a smaller company. The parent
> organization already has a mature data platform, but the acquired
> company's data does not yet follow the existing data pipeline and
> dimensional model.

Instead of creating a completely separate analytical platform, the
acquired company's data is onboarded into the existing architecture.

The implementation therefore focuses on:

1.  Building the missing ingestion and transformation pipelines.
2.  Standardizing the acquired company's data.
3.  Integrating its dimensions with the parent's existing dimensions.
4.  Loading historical fact data.
5.  Processing future changes incrementally.
6.  Automating the pipeline using Databricks Workflows.

This makes the project representative of a **real-world enterprise data
integration and acquisition onboarding scenario**.

------------------------------------------------------------------------

# 📊 Final Outcome

After successful execution:

-   Sportsbar raw data is available in Bronze.
-   Cleaned Sportsbar data is available in Silver.
-   Sportsbar dimension data is integrated into the parent Gold
    dimensions.
-   Historical Sportsbar facts are loaded through the full-load
    pipeline.
-   New/changed fact data is processed through the incremental pipeline.
-   Parent and child-company data are available through the integrated
    Gold model.
-   Databricks Workflows automate the complete processing.
-   The solution is scalable for onboarding additional acquired
    companies in the future.

------------------------------------------------------------------------

# 👨‍💻 Skills Demonstrated

**Data Engineering:**\
ETL/ELT • Data Integration • Data Quality • Data Modeling • Incremental
Processing

**Databricks:**\
Databricks Workflows • Notebooks • Unity Catalog • Widgets • Serverless
SQL/Compute

**Big Data:**\
PySpark • Spark SQL • Delta Lake

**Cloud:**\
Amazon S3 • External Locations • Cloud Data Engineering

**Architecture:**\
Medallion Architecture • Bronze/Silver/Gold • Parent-Child Data
Integration

------------------------------------------------------------------------

## ⭐ Project Highlights

-   Built an end-to-end Databricks pipeline for an acquired company.
-   Ingested source CSV data from Amazon S3.
-   Implemented Bronze → Silver → Gold processing.
-   Developed data cleansing and standardization rules.
-   Integrated child-company dimensions into existing parent Gold
    dimensions using Delta `MERGE`.
-   Developed separate full-load and incremental-load fact pipelines.
-   Implemented insert/update processing.
-   Parameterized notebooks using Databricks widgets.
-   Organized notebooks into a consolidated pipeline.
-   Automated daily incremental execution at 11:00 PM using Databricks Workflows.
-   Configured email notifications for relevant team members for job execution status.

------------------------------------------------------------------------

## 📌 Conclusion

The project demonstrates how a data engineering team can onboard an
acquired company's data into an existing enterprise data platform while
preserving the parent's architecture and analytical model.

The final solution provides a **scalable, maintainable, and automated
Databricks data pipeline** covering ingestion, transformation, dimension
integration, fact processing, incremental loading, and workflow
scheduling.
